# Warsaw Daily Weather Data - Group Preprocessing Pipeline
This notebook combines all individual preprocessing steps into one single workflow.

## 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler

# Set plot styles
sns.set_theme(style='whitegrid')
os.makedirs('results/eda_visualizations', exist_ok=True)
os.makedirs('results/outputs', exist_ok=True)

# Load data
df = pd.read_csv('data/raw/warsaw.csv')
print('Dataset loaded:', df.shape)


## 2. Missing Data Handling (IT25101533 - PERERA D.T.M)

In [ ]:
# Fill numerical columns with median
for column in ['PRCP', 'TMAX', 'TMIN', 'SNWD', 'TAVG']:
    if column in df.columns:
        df[column] = df[column].fillna(df[column].median())

print('Missing values after imputation:\n', df.isnull().sum())


## 3. Outlier Removal (IT25103406 - RATHNAYAKE R.M.M.M)

In [ ]:
def remove_outliers_iqr(data, columns):
    df_cleaned = data.copy()
    for col in columns:
        Q1 = df_cleaned[col].quantile(0.25)
        Q3 = df_cleaned[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        condition = ((df_cleaned[col] >= lower_bound) & (df_cleaned[col] <= upper_bound)) | df_cleaned[col].isna()
        df_cleaned = df_cleaned[condition]
    return df_cleaned

# Remove outliers from PRCP and TMAX
df = remove_outliers_iqr(df, ['PRCP', 'TMAX'])
print('Dataset shape after outlier removal:', df.shape)


## 4. Data Transformation (IT25102520 - NIRMAL W A D D T)

In [ ]:
# Apply log1p transformation to PRCP to handle skewness
df['PRCP_log'] = np.log1p(df['PRCP'])
print('Skewness of PRCP:', df['PRCP'].skew())
print('Skewness of PRCP_log:', df['PRCP_log'].skew())


## 5. Feature Encoding (IT25101708 - KURUPPU R.A.L)

In [ ]:
# Convert DATE to datetime
df['DATE'] = pd.to_datetime(df['DATE'])

def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

# Extract Season
df['Season'] = df['DATE'].dt.month.map(get_season)

# Apply One-Hot Encoding
df = pd.get_dummies(df, columns=['Season'], prefix='Season')
print('Columns after encoding:', df.columns.tolist())


## 6. Normalization & Scaling (IT25103602 - THIDASVIN R.K.P.L)

In [ ]:
scaler = StandardScaler()
features_to_scale = ['PRCP_log', 'SNWD', 'TMAX', 'TMIN', 'TAVG']
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])
print('Data scaled successfully.')


## 7. Feature Selection (IT25100613 - JAYATHILAKE K A V S)

In [ ]:
# Remove unneeded identifier and redundant columns
columns_to_drop = ['STATION', 'NAME', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'PRCP']
df_clean = df.drop(columns=columns_to_drop)
print('Final Dataset shape:', df_clean.shape)


## 8. Save Processed Dataset

In [ ]:
output_path = 'results/outputs/warsaw_processed.csv'
df_clean.to_csv(output_path, index=False)
print(f'Processed dataset saved to {output_path}')
